# con dos sensores

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_squared_error, mean_absolute_error
import warnings
warnings.filterwarnings('ignore')

class HybridTimeSeriesImputer:
    """
    Clase simplificada que usa ÚNICAMENTE el método híbrido para imputar datos faltantes
    Ahora soporta 4 sensores: temperatureC, humidityporc, temperatureC2, humidityporc2
    """

    def __init__(self, csv_file):
        self.csv_file = csv_file
        self.df = None
        self.df_complete = None
        self.missing_stats = {}
        # Definir las columnas de sensores
        self.sensor_columns = ['temperatureC', 'humidityporc', 'temperatureC2', 'humidityporc2']

    def load_and_prepare_data(self):
        """Carga y prepara los datos creando timestamp completo"""
        # Cargar datos
        self.df = pd.read_csv(self.csv_file)

        # Limpiar nombres de columnas (remover espacios)
        self.df.columns = self.df.columns.str.strip()

        # Crear datetime
        self.df['datetime'] = pd.to_datetime(
            self.df['date'] + ' ' +
            self.df['hours'].astype(str).str.zfill(2) + ':' +
            self.df['minutes'].astype(str).str.zfill(2)
        )

        # Ordenar por datetime
        self.df = self.df.sort_values('datetime').reset_index(drop=True)

        print(f"📁 Datos cargados: {len(self.df)} registros")
        print(f"🕒 Rango temporal: {self.df['datetime'].min()} a {self.df['datetime'].max()}")

        # Verificar qué columnas de sensores existen
        existing_sensors = [col for col in self.sensor_columns if col in self.df.columns]
        print(f"📊 Sensores detectados: {existing_sensors}")
        self.sensor_columns = existing_sensors  # Actualizar lista con sensores existentes

        return self.df

    def create_complete_timeline(self):
        """Crea timeline completo con intervalos de 1 minuto"""
        # Determinar rango temporal completo
        start_time = self.df['datetime'].min()
        end_time = self.df['datetime'].max()

        # Crear timeline completo por minutos
        complete_timeline = pd.date_range(
            start=start_time,
            end=end_time,
            freq='1T'  # 1 minuto
        )

        # Crear DataFrame completo
        df_complete = pd.DataFrame({'datetime': complete_timeline})

        # Preparar columnas para merge
        merge_columns = ['datetime'] + self.sensor_columns

        # Merge con datos existentes
        self.df_complete = df_complete.merge(
            self.df[merge_columns],
            on='datetime',
            how='left'
        )

        # Agregar columnas de tiempo para análisis
        self.df_complete['hour'] = self.df_complete['datetime'].dt.hour
        self.df_complete['minute'] = self.df_complete['datetime'].dt.minute
        self.df_complete['day'] = self.df_complete['datetime'].dt.day
        self.df_complete['time_numeric'] = (
            self.df_complete['datetime'] - self.df_complete['datetime'].min()
        ).dt.total_seconds() / 60  # minutos desde inicio

        print(f"📊 Timeline completo creado: {len(self.df_complete)} registros")

        return self.df_complete

    def analyze_missing_data(self):
        """Analiza patrones de datos faltantes para todos los sensores"""
        total_records = len(self.df_complete)

        self.missing_stats = {'total_records': total_records}

        print("\n=== 🔍 ANÁLISIS DE DATOS FALTANTES ===")
        print(f"📈 Total registros: {total_records}")

        # Analizar cada sensor
        for sensor in self.sensor_columns:
            missing_count = self.df_complete[sensor].isna().sum()
            missing_pct = (missing_count / total_records) * 100

            self.missing_stats[f'{sensor}_missing'] = missing_count
            self.missing_stats[f'{sensor}_missing_pct'] = missing_pct

            # Emoji según tipo de sensor
            emoji = "🌡️" if "temperature" in sensor else "💧"
            print(f"{emoji} {sensor} faltante: {missing_count} ({missing_pct:.2f}%)")

        # Analizar gaps consecutivos para el primer sensor de temperatura
        temp_sensor = next((s for s in self.sensor_columns if "temperature" in s), None)
        if temp_sensor:
            self.analyze_gaps(temp_sensor)

        return self.missing_stats

    def analyze_gaps(self, sensor_column):
        """Analiza la longitud de gaps consecutivos y su clasificación"""
        gaps = []
        gap_details = []

        # Encontrar gaps en el sensor especificado
        missing_mask = self.df_complete[sensor_column].isna()
        gap_starts = missing_mask & (~missing_mask.shift(1, fill_value=False))
        gap_ends = (~missing_mask) & missing_mask.shift(1, fill_value=False)

        gap_start_indices = self.df_complete[gap_starts].index
        gap_end_indices = self.df_complete[gap_ends].index

        if len(gap_start_indices) > len(gap_end_indices):
            gap_end_indices = np.append(gap_end_indices, len(self.df_complete))

        for start, end in zip(gap_start_indices, gap_end_indices):
            gap_size = end - start
            gaps.append(gap_size)

            # Clasificar gap según el método híbrido
            if gap_size <= 5:
                gap_type = "PEQUEÑO (Interpolación Lineal)"
            elif gap_size <= 30:
                gap_type = "MEDIANO (Interpolación Spline)"
            else:
                gap_type = "GRANDE (Promedio Estacional + Ruido)"

            gap_details.append({
                'inicio': start,
                'fin': end,
                'tamaño': gap_size,
                'tipo': gap_type
            })

        if gaps:
            print(f"\n🕳️ Análisis de Gaps ({sensor_column}):")
            print(f"  📊 Número total de gaps: {len(gaps)}")
            print(f"  📏 Gap promedio: {np.mean(gaps):.1f} minutos")
            print(f"  📈 Gap máximo: {max(gaps)} minutos")
            print(f"  📉 Gap mínimo: {min(gaps)} minutos")

            # Clasificar gaps por tipo
            small_gaps = sum(1 for g in gaps if g <= 5)
            medium_gaps = sum(1 for g in gaps if 5 < g <= 30)
            large_gaps = sum(1 for g in gaps if g > 30)

            print(f"\n🎯 Clasificación para Método Híbrido:")
            print(f"  🔹 Gaps pequeños (≤5 min): {small_gaps} - Usará interpolación lineal")
            print(f"  🔸 Gaps medianos (6-30 min): {medium_gaps} - Usará interpolación spline")
            print(f"  🔶 Gaps grandes (>30 min): {large_gaps} - Usará promedio estacional")

        return gap_details

    def method_hybrid_approach(self):
        """
        Método híbrido optimizado para ML aplicado a todos los sensores:
        - Gaps pequeños (≤5 min): Interpolación lineal
        - Gaps medianos (6-30 min): Interpolación spline
        - Gaps grandes (>30 min): Promedio estacional + ruido realista
        """
        print("\n=== 🔧 APLICANDO MÉTODO HÍBRIDO A TODOS LOS SENSORES ===")

        df_hybrid = self.df_complete.copy()

        # Configurar semilla para reproducibilidad
        np.random.seed(42)

        # Procesar cada sensor individualmente
        for sensor in self.sensor_columns:
            print(f"\n🔄 Procesando sensor: {sensor}")

            # Copiar serie original
            original_series = df_hybrid[sensor].copy()

            # Calcular estadísticas por hora como fallback para gaps grandes
            hourly_mean = df_hybrid.groupby('hour')[sensor].mean()
            hourly_std = df_hybrid.groupby('hour')[sensor].std().fillna(1.0 if 'temperature' in sensor else 5.0)

            # Aplicar interpolación lineal como base
            linear_series = original_series.interpolate(method='linear')

            # Aplicar interpolación spline para suavizar
            try:
                spline_series = original_series.interpolate(method='spline', order=3)
            except:
                spline_series = linear_series.copy()
                print(f"⚠️ Spline falló para {sensor}, usando interpolación lineal como fallback")

            # Identificar gaps y aplicar lógica híbrida
            missing_mask = original_series.isna()
            gap_starts = missing_mask & (~missing_mask.shift(1, fill_value=False))
            gap_start_indices = df_hybrid[gap_starts].index

            gaps_processed = {'small': 0, 'medium': 0, 'large': 0}

            # Inicializar serie híbrida
            hybrid_column = f'{sensor}_hybrid'
            df_hybrid[hybrid_column] = original_series.copy()

            # Procesar cada gap individualmente
            for start_idx in gap_start_indices:
                # Encontrar el final del gap
                end_idx = start_idx
                while end_idx < len(df_hybrid) and missing_mask.iloc[end_idx]:
                    end_idx += 1

                gap_size = end_idx - start_idx
                gap_indices = range(start_idx, end_idx)

                if gap_size <= 5:
                    # Gap pequeño: usar interpolación lineal
                    for idx in gap_indices:
                        df_hybrid.loc[idx, hybrid_column] = linear_series.iloc[idx]
                    gaps_processed['small'] += 1

                elif gap_size <= 30:
                    # Gap mediano: usar interpolación spline
                    for idx in gap_indices:
                        df_hybrid.loc[idx, hybrid_column] = spline_series.iloc[idx]
                    gaps_processed['medium'] += 1

                else:
                    # Gap grande: usar promedio estacional con variabilidad
                    for idx in gap_indices:
                        hour = df_hybrid.loc[idx, 'hour']

                        # Agregar ruido realista basado en la variabilidad histórica
                        noise_std = hourly_std[hour] * 0.3
                        noise = np.random.normal(0, noise_std)

                        # Usar promedio por hora + ruido
                        df_hybrid.loc[idx, hybrid_column] = hourly_mean[hour] + noise

                    gaps_processed['large'] += 1

            # Reporte de procesamiento por sensor
            print(f"  ✅ {sensor} - Gaps procesados: Pequeños={gaps_processed['small']}, Medianos={gaps_processed['medium']}, Grandes={gaps_processed['large']}")

            # Validar que no hay datos faltantes
            remaining_missing = df_hybrid[hybrid_column].isna().sum()
            if remaining_missing > 0:
                print(f"  ⚠️ Advertencia: Quedan {remaining_missing} valores faltantes en {sensor}")
            else:
                print(f"  ✅ {sensor} completamente imputado")

        print(f"\n🎯 RESUMEN GENERAL:")
        print(f"✅ Todos los sensores procesados exitosamente")

        return df_hybrid

    def export_clean_data(self, output_file='data_clean_hybrid_4sensors.csv'):
        """Exporta datos limpios usando el método híbrido para todos los sensores"""

        # Aplicar método híbrido
        df_hybrid = self.method_hybrid_approach()

        print(f"\n=== 💾 EXPORTANDO DATOS LIMPIOS ===")

        # Crear DataFrame limpio con estructura original
        clean_df = pd.DataFrame()

        # Extraer componentes de tiempo del datetime
        clean_df['date'] = df_hybrid['datetime'].dt.strftime('%Y-%m-%d')
        clean_df['hours'] = df_hybrid['datetime'].dt.hour
        clean_df['minutes'] = df_hybrid['datetime'].dt.minute

        # Agregar cada sensor imputado
        for sensor in self.sensor_columns:
            hybrid_column = f'{sensor}_hybrid'

            if 'temperature' in sensor:
                # Redondear temperatura a 1 decimal
                clean_df[sensor] = df_hybrid[hybrid_column].round(1)
            else:
                # Convertir humedad a entero (redondeando)
                clean_df[sensor] = df_hybrid[hybrid_column].round().astype(int)

        # Verificar si existe columna 'id' en datos originales
        if 'id' in self.df.columns:
            # Si existe, usar el rango completo de IDs
            start_id = self.df['id'].min() if not self.df['id'].isna().all() else 1
            clean_df.insert(0, 'id', range(start_id, start_id + len(clean_df)))

        # Exportar archivo
        clean_df.to_csv(output_file, index=False)

        # Reporte final
        print(f"✅ Datos exportados exitosamente:")
        print(f"  📄 Archivo: {output_file}")
        print(f"  📊 Total registros: {len(clean_df)}")
        print(f"  🎯 Método usado: HÍBRIDO")
        print(f"  ❌ Datos faltantes: {clean_df.isnull().sum().sum()}")
        print(f"  📋 Estructura: {list(clean_df.columns)}")

        # Mostrar tipos de datos
        data_types = []
        for col in clean_df.columns:
            if col in ['date', 'hours', 'minutes', 'id']:
                continue
            elif 'temperature' in col:
                data_types.append(f"{col}=float(1 decimal)")
            elif 'humidity' in col:
                data_types.append(f"{col}=integer")
        print(f"  🔢 Tipos: {', '.join(data_types)}")

        # Estadísticas finales por sensor
        print(f"\n📈 Estadísticas finales:")
        for sensor in self.sensor_columns:
            if sensor in clean_df.columns:
                if 'temperature' in sensor:
                    min_val = clean_df[sensor].min()
                    max_val = clean_df[sensor].max()
                    print(f"  🌡️ {sensor}: {min_val:.1f}°C - {max_val:.1f}°C")
                else:
                    min_val = clean_df[sensor].min()
                    max_val = clean_df[sensor].max()
                    print(f"  💧 {sensor}: {min_val}% - {max_val}%")

        return clean_df

    def visualize_results(self, clean_df, sample_size=200):
        """Visualiza los resultados del método híbrido para todos los sensores"""

        # Crear DataFrame para visualización
        viz_df = self.df_complete.copy()

        # Agregar datos imputados
        for sensor in self.sensor_columns:
            viz_df[f'{sensor}_hybrid'] = clean_df[sensor].values

        # Tomar muestra si el dataset es muy grande
        if len(viz_df) > sample_size:
            viz_df = viz_df.sample(n=sample_size).sort_values('datetime')

        # Determinar número de filas para subplots
        n_sensors = len(self.sensor_columns)
        fig, axes = plt.subplots(n_sensors, 1, figsize=(15, 4 * n_sensors))

        # Si solo hay un sensor, convertir axes a lista
        if n_sensors == 1:
            axes = [axes]

        # Colores para cada tipo de sensor
        colors = {'temperature': 'blue', 'humidity': 'green'}

        # Crear gráfico para cada sensor
        for i, sensor in enumerate(self.sensor_columns):
            # Determinar color según tipo de sensor
            color = colors.get('temperature' if 'temperature' in sensor else 'humidity', 'blue')

            # Determinar emoji y unidad
            if 'temperature' in sensor:
                emoji = "🌡️"
                unit = "°C"
            else:
                emoji = "💧"
                unit = "%"

            # Plotear datos originales y imputados
            axes[i].plot(viz_df['datetime'], viz_df[sensor], 'ro',
                        label='Datos Originales', markersize=4, alpha=0.7)
            axes[i].plot(viz_df['datetime'], viz_df[f'{sensor}_hybrid'], f'{color[0]}-',
                        label='Método Híbrido', linewidth=2, alpha=0.8)

            axes[i].set_title(f'{emoji} Imputación de {sensor} - Método Híbrido', fontsize=14)
            axes[i].set_ylabel(f'{sensor} ({unit})')
            axes[i].legend()
            axes[i].grid(True, alpha=0.3)

            # Solo agregar xlabel en el último subplot
            if i == len(self.sensor_columns) - 1:
                axes[i].set_xlabel('Tiempo')

        plt.tight_layout()
        plt.show()

        print("📊 Visualización generada - puntos rojos son datos originales, líneas son datos imputados")

# Función principal simplificada
def process_with_hybrid_only(csv_file, output_file='data_clean_hybrid_4sensors.csv', visualize=False):
    """
    Función principal que usa ÚNICAMENTE el método híbrido para todos los sensores

    Args:
        csv_file: Archivo CSV de entrada
        output_file: Archivo CSV de salida
        visualize: Si mostrar gráficos (default False)
    """
    print("🚀 PROCESAMIENTO CON MÉTODO HÍBRIDO - 4 SENSORES")
    print("=" * 60)

    # Crear instancia del imputador híbrido
    imputer = HybridTimeSeriesImputer(csv_file)

    # Procesar datos paso a paso
    imputer.load_and_prepare_data()
    imputer.create_complete_timeline()
    imputer.analyze_missing_data()

    # Aplicar método híbrido y exportar
    clean_data = imputer.export_clean_data(output_file)

    # Visualizar si se solicita
    if visualize:
        imputer.visualize_results(clean_data)

    print(f"\n🎯 RESUMEN FINAL:")
    print(f"✅ Método usado: HÍBRIDO (inteligente según tamaño de gap)")
    print(f"✅ Sensores procesados: {imputer.sensor_columns}")
    print(f"✅ Dataset completamente imputado y listo para ML")
    print(f"✅ Archivo guardado: {output_file}")
    print(f"✅ Estructura original mantenida para compatibilidad")

    print(f"\n🧠 LÓGICA DEL MÉTODO HÍBRIDO:")
    print(f"  🔹 Gaps ≤5 min → Interpolación lineal (preserva tendencias)")
    print(f"  🔸 Gaps 6-30 min → Interpolación spline (transiciones suaves)")
    print(f"  🔶 Gaps >30 min → Promedio estacional + ruido (patrones naturales)")

    return clean_data, imputer

# Ejemplo de uso
if __name__ == "__main__":
    # Procesar con método híbrido para 4 sensores
    clean_data, imputer = process_with_hybrid_only(
        csv_file='/content/drive/MyDrive/nice/ALLdatanew/dataclean_1year.csv',  # Ajusta la ruta según tu archivo
        output_file='/content/drive/MyDrive/nice/ALLdatanew/dataclean_1year.csv',
        visualize=True  # Cambiar a False si no quieres gráficos
    )

    # Mostrar muestra de datos limpios
    print("\n📊 MUESTRA DE DATOS LIMPIOS:")
    print(clean_data.head(10))
    print("\n📈 ESTADÍSTICAS FINALES:")
    sensor_cols = [col for col in clean_data.columns if col in ['temperatureC', 'humidityporc', 'temperatureC2', 'humidityporc2']]
    print(clean_data[sensor_cols].describe())